In [ ]:
import pandas as pd

from statx_helpers import (
    DATASET_SPECS,
    TARGET_COL,
    fit_and_score_dataset,
    make_dataset,
    split_by_period,
)

input_files = {
    "all_drugs": "../outputs/train_all_drugs_merged.csv",
    "all_opioids": "../outputs/train_all_opioids_merged.csv",
    "all_stimulants": "../outputs/train_all_stimulants_merged.csv",
}


In [ ]:
results = []

for prediction_type, path in input_files.items():
    print("\n============================================================")
    print(f"Prediction type: {prediction_type}")
    print("============================================================")

    raw_df = pd.read_csv(path).dropna(subset=[TARGET_COL])
    train_raw, val_raw, train_periods, val_periods = split_by_period(
        raw_df,
        val_size=0.2,
        random_state=42,
    )
    print("Train periods:", len(train_periods), "| Validation periods:", len(val_periods))

    for dataset_name, spec in DATASET_SPECS.items():
        print(f"\n-------------------- {prediction_type} | {dataset_name} --------------------")

        train_df = make_dataset(train_raw, spec["method"]).drop(columns=["overdose_category"], errors="ignore")
        val_df = make_dataset(val_raw, spec["method"], reference=train_raw).drop(columns=["overdose_category"], errors="ignore")

        results.extend(
            fit_and_score_dataset(
                train_df,
                val_df,
                dataset_name=dataset_name,
                prediction_type=prediction_type,
            )
        )


In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values(by=["prediction_type", "RMSE"], ascending=[True, True])
    .reset_index(drop=True)
)

results_df


In [ ]:
results_df.to_csv(
    "../outputs/category_period_model_comparison.csv",
    index=False,
)

print("Saved:")
print("../outputs/category_period_model_comparison.csv")


In [ ]:
best_by_type = (
    results_df
    .sort_values("RMSE")
    .groupby("prediction_type")
    .head(1)
    .reset_index(drop=True)
)

best_by_type
